Load data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from pmdarima import auto_arima
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

In [ ]:
df = pd.read_csv("AirPassengers.csv")
df['Month'] = pd.to_datetime(df['Month'])
df.set_index('Month', inplace=True)

=== CHAPTER 2: CLEANING AND PREPROCESSING ===

In [ ]:
df['Month_num'] = df.index.month
df['Year'] = df.index.year
df['TimeIndex'] = np.arange(len(df))

Check for nulls and duplicates

In [ ]:
print("Null values:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

Train-test split (80-20)

In [ ]:
train = df[:'1959']
test = df['1960':]

=== CHAPTER 3: EDA ===

In [ ]:
print("Descriptive Stats:\n", df['#Passengers'].describe())
print("Skewness:", df['#Passengers'].skew())

Line plot

In [ ]:
df['#Passengers'].plot(title='Monthly Passengers Over Time', figsize=(10,5))
plt.xlabel('Date'); plt.ylabel('Passengers'); plt.show()

istogram

In [ ]:
df['#Passengers'].hist(bins=12, edgecolor='black')
plt.title("Histogram of Passengers"); plt.show()

Boxplot by Month

In [ ]:
sns.boxplot(x='Month_num', y='#Passengers', data=df)
plt.title("Boxplot by Month"); plt.show()

Correlation Heatmap

In [ ]:
sns.heatmap(df.corr(), annot=True, cmap="coolwarm"); plt.title("Correlation Matrix"); plt.show()

Seasonal Decomposition

In [ ]:
result = seasonal_decompose(df['#Passengers'], model='multiplicative')
result.plot(); plt.show()

=== CHAPTER 4: MODELING ===

--- ARIMA ---

In [ ]:
model_arima = auto_arima(train['#Passengers'], seasonal=True, m=12, trace=False)
forecast_arima = model_arima.predict(n_periods=len(test))

Plot ARIMA forecast

In [ ]:
plt.plot(train.index, train['#Passengers'], label='Train')
plt.plot(test.index, test['#Passengers'], label='Test')
plt.plot(test.index, forecast_arima, label='ARIMA Forecast', linestyle='--')
plt.title("ARIMA Forecast vs Actual")
plt.legend(); plt.show()

--- LINEAR REGRESSION ---

In [ ]:
lr = LinearRegression()
lr.fit(train['TimeIndex'].values.reshape(-1, 1), train['#Passengers'])
forecast_lr = lr.predict(test['TimeIndex'].values.reshape(-1, 1))

In [ ]:
plt.plot(train.index, train['#Passengers'], label='Train')
plt.plot(test.index, test['#Passengers'], label='Test')
plt.plot(test.index, forecast_lr, label='Linear Regression Forecast', linestyle='--')
plt.title("Linear Regression Forecast")
plt.legend(); plt.show()

--- POLYNOMIAL REGRESSION ---

In [ ]:
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(train['TimeIndex'].values.reshape(-1, 1))
lr_poly = LinearRegression()
lr_poly.fit(X_poly, train['#Passengers'])

In [ ]:
X_test_poly = poly.transform(test['TimeIndex'].values.reshape(-1, 1))
forecast_poly = lr_poly.predict(X_test_poly)

In [ ]:
plt.plot(train.index, train['#Passengers'], label='Train')
plt.plot(test.index, test['#Passengers'], label='Test')
plt.plot(test.index, forecast_poly, label='Poly Regression Forecast', linestyle='--')
plt.title("Polynomial Regression Forecast")
plt.legend(); plt.show()

=== CHAPTER 5: EVALUATION ===

ARIMA metrics

In [ ]:
rmse_arima = np.sqrt(mean_squared_error(test['#Passengers'], forecast_arima))
mape_arima = mean_absolute_percentage_error(test['#Passengers'], forecast_arima)
r2_arima = r2_score(test['#Passengers'], forecast_arima)

Linear Regression metrics

In [ ]:
rmse_lr = np.sqrt(mean_squared_error(test['#Passengers'], forecast_lr))
mape_lr = mean_absolute_percentage_error(test['#Passengers'], forecast_lr)
r2_lr = r2_score(test['#Passengers'], forecast_lr)

Polynomial Regression metrics

In [ ]:
rmse_poly = np.sqrt(mean_squared_error(test['#Passengers'], forecast_poly))
mape_poly = mean_absolute_percentage_error(test['#Passengers'], forecast_poly)
r2_poly = r2_score(test['#Passengers'], forecast_poly)

Compare results

In [ ]:
print("\nModel Evaluation Results:")
print("ARIMA    â†’ RMSE: %.2f | MAPE: %.2f | RÂ²: %.2f" % (rmse_arima, mape_arima, r2_arima))
print("Linear   â†’ RMSE: %.2f | MAPE: %.2f | RÂ²: %.2f" % (rmse_lr, mape_lr, r2_lr))
print("PolyReg  â†’ RMSE: %.2f | MAPE: %.2f | RÂ²: %.2f" % (rmse_poly, mape_poly, r2_poly))